# Classification binaire de la durée d’hospitalisation : XGBoost vs LightGBM

## Objectif
Prédire un séjour long (7 jours ou plus) sur `diabetic_data.csv` et comparer XGBoost non équilibré / équilibré à un LightGBM de référence.

## Protocole
- Séparation train/test par patient avec `StratifiedGroupKFold` (même graine et même split pour tous les modèles).
- `time_in_hospital`, les identifiants et certaines variables disponibles à la sortie sont exclus des prédicteurs.
- XGBoost utilise les variables catégorielles natives (`enable_categorical=True`, `tree_method='hist'`).
- Équilibrage XGBoost via `scale_pos_weight = nombre de négatifs / nombre de positifs` calculé sur le train uniquement.
- Comparaison des seuils 0,50 et 0,27 ; le jeu de test n’est jamais rééquilibré.

**Attention :** seuils 0,50 et 0,27 sont comparés à titre expérimental. Le seuil final devrait être choisi sur un jeu de validation, pas sur le test. Ce modèle est exploratoire et ne convient pas à une décision médicale réelle sans validation externe.

## 1. Entraînement et évaluation

In [1]:
import os, time, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
RANDOM_STATE=42
CSV=Path('diabetic_data.csv')
OUT=Path('resultats_xgboost_binaire'); OUT.mkdir(exist_ok=True)
df=pd.read_csv(CSV, na_values=['?'], keep_default_na=False, low_memory=False)
stay_days=pd.to_numeric(df['time_in_hospital'], errors='raise').astype(int)
y=(stay_days>=7).astype(int)
stratification_labels=pd.Series(np.select([stay_days<=3,stay_days<=6],[0,1],default=2), index=df.index).astype(int)
# Same leakage controls and preprocessing as prior lab: target and post-stay columns excluded.
exclude=['encounter_id','patient_nbr','time_in_hospital','discharge_disposition_id','readmitted','weight']
work=df.copy()
if 'age' in work:
    age_bounds=work['age'].astype('string').str.extract(r'\[(\d+)-(\d+)\)')
    work['age_num']=age_bounds.astype(float).mean(axis=1)
    work=work.drop(columns=['age'])
X=work.drop(columns=[c for c in exclude if c in work.columns]).copy()
constant=[c for c in X.columns if X[c].nunique(dropna=False)<=1]
X=X.drop(columns=constant)
num_cols=['age_num','num_lab_procedures','num_procedures','num_medications','number_outpatient','number_emergency','number_inpatient','number_diagnoses']
num_cols=[c for c in num_cols if c in X.columns]
for c in num_cols: X[c]=pd.to_numeric(X[c],errors='coerce')
cat_cols=[c for c in X.columns if c not in num_cols]
for c in cat_cols: X[c]=X[c].astype('string').fillna('Inconnu').replace({'?':'Inconnu'}).astype(str).astype('category')
X=X.reset_index(drop=True); y=y.reset_index(drop=True); stratification_labels=stratification_labels.reset_index(drop=True); groups=df['patient_nbr'].reset_index(drop=True)
splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
train_idx,test_idx=next(splitter.split(X,stratification_labels,groups=groups))
X_train=X.iloc[train_idx].reset_index(drop=True); X_test=X.iloc[test_idx].reset_index(drop=True)
y_train=y.iloc[train_idx].reset_index(drop=True); y_test=y.iloc[test_idx].reset_index(drop=True)
assert not (set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
# Class balancing ratio for XGBoost.
scale_pos=float((y_train==0).sum()/(y_train==1).sum())
print('Dataset',df.shape,'train/test',len(X_train),len(X_test),'positive ratio train',y_train.mean(),'scale_pos_weight',scale_pos,flush=True)
# XGBoost categorical-native models, same params except scale_pos_weight.
base_params=dict(n_estimators=350, max_depth=6, learning_rate=0.05, min_child_weight=5,
                 subsample=0.85, colsample_bytree=0.85, reg_lambda=3.0,
                 objective='binary:logistic', eval_metric='logloss', tree_method='hist',
                 enable_categorical=True, n_jobs=min(4,os.cpu_count() or 2), random_state=RANDOM_STATE)
models={}
for name,weight in [('XGBoost sans équilibrage',1.0),('XGBoost équilibré',scale_pos)]:
    print('Training',name,flush=True); t=time.time()
    model=XGBClassifier(**base_params,scale_pos_weight=weight)
    model.fit(X_train,y_train,verbose=False)
    models[name]=model
    print('Done',name,'minutes',round((time.time()-t)/60,2),flush=True)
# Also fit LightGBM unweighted on identical split for fair comparison.
print('Training LightGBM reference',flush=True); t=time.time()
lgb=LGBMClassifier(n_estimators=180,learning_rate=.05,num_leaves=31,max_depth=-1,min_child_samples=50,
    colsample_bytree=.9,reg_lambda=3.0,subsample=.9,subsample_freq=1,max_bin=127,
    random_state=RANDOM_STATE,n_jobs=min(4,os.cpu_count() or 2),verbosity=-1)
lgb.fit(X_train,y_train)
models['LightGBM sans équilibrage']=lgb
print('Done LightGBM minutes',round((time.time()-t)/60,2),flush=True)

def metrics(name, ytrue, prob, threshold):
    pred=(prob>=threshold).astype(int)
    return {'Modèle':name,'Seuil':threshold,'Accuracy':accuracy_score(ytrue,pred),
            'Balanced accuracy':balanced_accuracy_score(ytrue,pred),
            'Précision séjour long':precision_score(ytrue,pred,zero_division=0),
            'Rappel séjour long':recall_score(ytrue,pred,zero_division=0),
            'F1 séjour long':f1_score(ytrue,pred,zero_division=0),
            'Macro-F1':f1_score(ytrue,pred,average='macro',zero_division=0),
            'ROC-AUC':roc_auc_score(ytrue,prob),'PR-AUC':average_precision_score(ytrue,prob),
            'Prédictions positives (%)':100*pred.mean()}
rows=[]; probas={}
for name,model in models.items():
    p=model.predict_proba(X_test)[:,1]; probas[name]=p
    for threshold in [.5,.27]: rows.append(metrics(name,y_test,p,threshold))
results=pd.DataFrame(rows)
results.to_csv(OUT/'metriques_xgboost_comparaison.csv',index=False)
# store prediction probabilities, classifications and test truth
pred=pd.DataFrame({'y_true':y_test.to_numpy()})
for name,p in probas.items():
    safe=name.lower().replace(' ','_').replace('é','e')
    pred[f'proba_{safe}']=p
    pred[f'pred_{safe}_050']=(p>=.5).astype(int)
    pred[f'pred_{safe}_027']=(p>=.27).astype(int)
pred.to_csv(OUT/'predictions_test.csv',index=False)
# summary with actual results and metadata
meta={'rows':len(df),'columns':len(df.columns),'train_rows':len(X_train),'test_rows':len(X_test),
      'train_unique_patients':int(groups.iloc[train_idx].nunique()),'test_unique_patients':int(groups.iloc[test_idx].nunique()),
      'positive_rate_test':float(y_test.mean()),'scale_pos_weight':scale_pos,'n_features':int(X.shape[1]),
      'xgboost_version':__import__('xgboost').__version__,'lightgbm_version':__import__('lightgbm').__version__,
      'params':base_params}
(OUT/'metadata.json').write_text(json.dumps(meta,indent=2))
print('\nRESULTS\n',results.round(4).to_string(index=False),flush=True)
print('\nClassification report XGB unbalanced threshold .27\n',classification_report(y_test,(probas['XGBoost sans équilibrage']>=.27).astype(int),digits=4),flush=True)
print('\nConfusion matrices at threshold .27')
for name,p in probas.items(): print(name,confusion_matrix(y_test,(p>=.27).astype(int)).tolist())
# persist models for future evaluation
import joblib
joblib.dump({'xgboost_unbalanced':models['XGBoost sans équilibrage'],'xgboost_balanced':models['XGBoost équilibré'],
             'lightgbm_unbalanced':lgb,'threshold_reference':.27,'feature_columns':list(X.columns),
             'numeric_columns':num_cols,'categorical_columns':cat_cols,'scale_pos_weight':scale_pos,
             'random_state':RANDOM_STATE},OUT/'modeles_comparaison.joblib')
print('OUT',OUT,flush=True)


Dataset (101766, 50) train/test 81414 20352 positive ratio train 0.20782666371877073 scale_pos_weight 3.8117021276595744


Training XGBoost sans équilibrage


Done XGBoost sans équilibrage minutes 0.06


Training XGBoost équilibré


Done XGBoost équilibré minutes 0.07


Training LightGBM reference


Done LightGBM minutes 0.03



RESULTS
                    Modèle  Seuil  Accuracy  Balanced accuracy  Précision séjour long  Rappel séjour long  F1 séjour long  Macro-F1  ROC-AUC  PR-AUC  Prédictions positives (%)
 XGBoost sans équilibrage   0.50    0.8431             0.6993                 0.6851              0.4533          0.5456    0.7254   0.8599  0.6448                    13.7480
 XGBoost sans équilibrage   0.27    0.8094             0.7681                 0.5314              0.6976          0.6033    0.7389   0.8599  0.6448                    27.2750
        XGBoost équilibré   0.50    0.7950             0.7692                 0.5046              0.7250          0.5951    0.7289   0.8573  0.6392                    29.8546
        XGBoost équilibré   0.27    0.6736             0.7540                 0.3787              0.8915          0.5316    0.6406   0.8573  0.6392                    48.9092
LightGBM sans équilibrage   0.50    0.8428             0.6997                 0.6825              0.4550          0


Classification report XGB unbalanced threshold .27
               precision    recall  f1-score   support

           0     0.9136    0.8387    0.8745     16123
           1     0.5314    0.6976    0.6033      4229

    accuracy                         0.8094     20352
   macro avg     0.7225    0.7681    0.7389     20352
weighted avg     0.8342    0.8094    0.8182     20352




Confusion matrices at threshold .27
XGBoost sans équilibrage [[13522, 2601], [1279, 2950]]
XGBoost équilibré [[9939, 6184], [459, 3770]]
LightGBM sans équilibrage [[13460, 2663], [1296, 2933]]
OUT resultats_xgboost_binaire


## 2. Interprétation des résultats

À lire dans le tableau affiché ci-dessus :
- **Rappel** : proportion des vrais séjours longs détectés.
- **Précision** : proportion des alertes « séjour long » qui sont correctes.
- **F1** : compromis entre précision et rappel.
- **Balanced accuracy** et **Macro-F1** : évaluent les deux classes en tenant compte du déséquilibre.
- **PR-AUC** : capacité de classement pour la classe positive, indépendamment du seuil.

L’équilibrage peut augmenter le rappel mais aussi déclencher beaucoup plus de fausses alertes. Il faut donc choisir la méthode selon le coût métier des faux négatifs et faux positifs.

## 3. Fichiers générés
Le code exporte les métriques (`metriques_xgboost_comparaison.csv`), les prédictions du test (`predictions_test.csv`), les métadonnées et les modèles dans le dossier `resultats_xgboost_binaire`.

### Références
- [XGBoost — paramètres de XGBClassifier](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier)
- [XGBoost — support des variables catégorielles](https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html)
- [Scikit-learn — StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)